# MIST · Attention Gates + Boundary Loss — RunPod Training
**Branch**: `gates_boundary_implementation` | **Model**: `MIST_CAM` with Attention Gates

**Architecture vs. baseline (`MIST_CAM` on `main`)**:
- `Block_decoder` and `Block_decoder1`: gated skip connections via `AttentionGate` (uses decoder state to score each encoder skip position before concatenation)
- SSAM (`CBAM`) active in `Transformer` — matches paper Eq 11: `SSAM(X''') + X''' + X''`
- Loss: `0.7 × Dice + 0.3 × CE + 0.2 × BoundaryLoss` per powerset subset (BoundaryLoss = boundary-weighted CE, boundary pixels up-weighted by `w=3`)

**Training matches paper Section 3.7**: AdamW lr=1e-4, weight_decay=1e-4, batch=12, img=256×256, 300 epochs, fixed LR, powerset mutation on 4 decoder outputs.

**Run cells in order. No kernel restart needed.**

In [ ]:
# Cell 2 — GPU check
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'No GPU detected!'
print('GPU check passed')

In [ ]:
# Cell 3 — Install dependencies
# No mamba-ssm needed for this variant.
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args),
                       capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-600:]

pkgs = [
    'scipy>=1.14.0',
    'timm==0.9.12',
    'SimpleITK',
    'scikit-image',
    'einops',
    'tensorboardX',
    'tqdm',
    'gdown',
    'matplotlib',
]
for spec in pkgs:
    code, out = pip('install', spec, '--quiet')
    status = 'OK    ' if code == 0 else 'FAILED'
    print(f'  {status}  {spec}')
    if code != 0:
        print(out)

print('\nAll dependencies installed.')

In [ ]:
# Cell 4 — Smoke test imports
import numpy as np
import scipy
import torch
import timm

print(f'numpy     {np.__version__}')
print(f'scipy     {scipy.__version__}')
print(f'torch     {torch.__version__}  (CUDA {torch.version.cuda})')
print(f'timm      {timm.__version__}')
print('All imports OK')

In [ ]:
# Cell 5 — Clone / update repo (branch: gates_boundary_implementation)
import subprocess, os, sys

REPO_URL = 'https://github.com/biancafabian/MIST.git'
BRANCH   = 'gates_boundary_implementation'
REPO_DIR = '/workspace/MIST'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'Repo exists at {REPO_DIR}, syncing {BRANCH}...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH],
                   capture_output=True)
    r = subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard',
                        f'origin/{BRANCH}'], capture_output=True, text=True)
    print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
else:
    print(f'Cloning {BRANCH}...')
    r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    print(r.stdout.strip(), r.stderr.strip())
    assert r.returncode == 0, f'git clone failed: {r.stderr}'

# Verify key files exist
for fname in ['lib/MIST.py', 'lib/networks.py', 'ACDC_train_test.py',
              'utils/utils.py', 'utils/dataset_ACDC.py']:
    path = os.path.join(REPO_DIR, fname)
    ok = 'OK     ' if os.path.exists(path) else 'MISSING'
    print(f'  {ok}  {fname}')

# Confirm branch-specific additions are present
with open(os.path.join(REPO_DIR, 'lib', 'MIST.py')) as fh:
    mist_src = fh.read()
assert 'class AttentionGate' in mist_src, 'AttentionGate not found in lib/MIST.py — wrong branch?'
assert 'self.ssam' in mist_src,           'SSAM not active in lib/MIST.py — check Transformer class'
print('AttentionGate confirmed in lib/MIST.py')
print('SSAM (self.ssam) confirmed in lib/MIST.py')

with open(os.path.join(REPO_DIR, 'utils', 'utils.py')) as fh:
    utils_src = fh.read()
assert 'class BoundaryLoss' in utils_src, 'BoundaryLoss not found in utils/utils.py — wrong branch?'
assert 'from medpy' not in utils_src,     'medpy import still present in utils/utils.py!'
print('BoundaryLoss confirmed in utils/utils.py')
print('medpy-free confirmed')

with open(os.path.join(REPO_DIR, 'lib', 'networks.py')) as fh:
    nets_src = fh.read()
assert 'class MIST_CAM' in nets_src, 'MIST_CAM not found in lib/networks.py!'
print('MIST_CAM confirmed in lib/networks.py')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'cwd: {os.getcwd()}')

In [ ]:
# Cell 7 — Download ACDC dataset (skip if already present)
import os, subprocess, sys, zipfile

DATA_ROOT = './data/ACDC'

if (os.path.isdir(DATA_ROOT) and
        os.path.isdir(os.path.join(DATA_ROOT, 'train')) and
        len(os.listdir(os.path.join(DATA_ROOT, 'train'))) > 0):
    print(f'ACDC data found at {DATA_ROOT}, skipping download.')
else:
    ACDC_FILE_ID = 'YOUR_GDRIVE_FILE_ID_HERE'   # <-- REPLACE THIS
    assert ACDC_FILE_ID != 'YOUR_GDRIVE_FILE_ID_HERE', \
        'Set ACDC_FILE_ID to the Google Drive file ID before running!'

    import gdown
    print('Downloading ACDC dataset from Google Drive...')
    gdown.download(f'https://drive.google.com/uc?id={ACDC_FILE_ID}',
                   'ACDC_dataset.zip', quiet=False)
    print('Extracting...')
    with zipfile.ZipFile('ACDC_dataset.zip', 'r') as z:
        z.extractall('.')
    os.remove('ACDC_dataset.zip')
    print('Done.')

print('\nData layout check:')
for sub in ['train', 'valid', 'test', 'lists_ACDC']:
    p = os.path.join(DATA_ROOT, sub)
    if os.path.isdir(p):
        n = len(os.listdir(p))
        print(f'  {sub:15s}  {n} entries')
    else:
        print(f'  {sub:15s}  MISSING')

In [ ]:
# Cell 8 — Verify dataset
import os, glob, numpy as np

TRAIN_DIR = './data/ACDC/train'
TEST_DIR  = './data/ACDC/test'

train_files = glob.glob(os.path.join(TRAIN_DIR, '*.npz'))
test_files  = glob.glob(os.path.join(TEST_DIR,  '*.npz'))
print(f'Train slices : {len(train_files)}')
print(f'Test  slices : {len(test_files)}')

if train_files:
    d = np.load(train_files[0])
    print(f'Sample train  image={d["image"].shape}  label={d["label"].shape}  '
          f'classes={sorted(set(d["label"].ravel().tolist()))}')
if test_files:
    d = np.load(test_files[0])
    print(f'Sample test   image={d["image"].shape}  label={d["label"].shape}  '
          f'classes={sorted(set(d["label"].ravel().tolist()))}')
print('Dataset OK')

In [ ]:
# Cell 9 — Pre-download MaxViT pretrained weights
# networks.py will also auto-download if missing, but doing it here
# avoids a mid-training interruption on slow connections.
import os, torch

WEIGHTS_PATH = './pretrained_pth/maxvit/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth'
WEIGHTS_URL  = ('https://github.com/rwightman/pytorch-image-models/releases/'
                'download/v0.1-weights-maxx/maxxvit_rmlp_small_rw_256_sw-37e217ff.pth')

if os.path.exists(WEIGHTS_PATH):
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Weights already present ({size_mb:.0f} MB): {WEIGHTS_PATH}')
else:
    os.makedirs(os.path.dirname(WEIGHTS_PATH), exist_ok=True)
    print('Downloading MaxViT weights...')
    torch.hub.download_url_to_file(WEIGHTS_URL, WEIGHTS_PATH)
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
    print(f'Saved ({size_mb:.0f} MB): {WEIGHTS_PATH}')
print('MaxViT weights OK')

In [ ]:
# Cell 11 — Training configuration
# Matches paper Section 3.7: AdamW lr=1e-4, wd=1e-4, batch=12, img=256, 300 epochs, fixed LR.
BATCH_SIZE     = 12
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
MAX_EPOCHS     = 300
IMG_SIZE       = 256
NUM_CLASSES    = 4       # BG, RV, Myo, LV
SEED           = 2222
BOUNDARY_W     = 3       # boundary pixel up-weight in BoundaryLoss
LC1, LC2, LC3  = 0.7, 0.3, 0.2  # loss weights: LC1*Dice + LC2*CE + LC3*BoundaryLoss

DATA_DIR  = './data/ACDC'
LIST_DIR  = './data/ACDC/lists_ACDC'
TEST_DIR  = './data/ACDC/test'
SAVE_DIR  = './model_pth'

print(f'batch={BATCH_SIZE}  lr={LR}  wd={WEIGHT_DECAY}  epochs={MAX_EPOCHS}')
print(f'img={IMG_SIZE}  classes={NUM_CLASSES}  seed={SEED}')
print(f'loss={LC1}*Dice + {LC2}*CE + {LC3}*BoundaryLoss(w={BOUNDARY_W})  (powerset mutation)')

In [ ]:
# Cell 12 — Initialise model, data loaders, losses, optimiser
import os, sys, time, random
import numpy as np
import torch
import torch.optim as optim
from torch.nn.modules.loss import CrossEntropyLoss
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler, autocast
from scipy.ndimage import zoom

REPO_DIR = '/workspace/MIST'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)

from lib.networks import MIST_CAM
from utils.utils import DiceLoss, BoundaryLoss, powerset, _dc
from utils.dataset_ACDC import ACDCdataset, RandomGenerator

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Snapshot directory
run_id        = time.strftime('%H%M%S')
snapshot_path = os.path.join(SAVE_DIR, f'MIST_CAM_AG_BndLoss_{IMG_SIZE}_run{run_id}')
os.makedirs(snapshot_path, exist_ok=True)
print(f'Snapshot dir: {snapshot_path}')

# Model (MaxViT encoder + CAM decoder with AttentionGates + SSAM)
net = MIST_CAM(
    n_class=NUM_CLASSES,
    img_size_s1=(IMG_SIZE, IMG_SIZE),
    img_size_s2=(224, 224),
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
).cuda()
n_params = sum(p.numel() for p in net.parameters()) / 1e6
print(f'Model: MIST_CAM (AttentionGates + SSAM)  params={n_params:.1f}M')

# Losses
ce_loss       = CrossEntropyLoss()
dice_loss     = DiceLoss(NUM_CLASSES)
boundary_loss = BoundaryLoss(NUM_CLASSES, w=BOUNDARY_W)

# Data loaders
train_dataset = ACDCdataset(
    DATA_DIR, LIST_DIR, split='train',
    transform=transforms.Compose([RandomGenerator(output_size=[IMG_SIZE, IMG_SIZE])]))
val_dataset   = ACDCdataset(DATA_DIR, LIST_DIR, split='valid')
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=4, pin_memory=True)
valloader     = DataLoader(val_dataset, batch_size=1, shuffle=False)
print(f'Train: {len(train_dataset)} slices  |  Val: {len(val_dataset)} slices')
print(f'Train iters/epoch: {len(train_loader)}')

# Optimiser — paper Section 3.7: AdamW, fixed LR (no schedule)
optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = GradScaler()

# Save run config
with open(os.path.join(snapshot_path, 'config.txt'), 'w') as f:
    f.write('model=MIST_CAM\n')
    f.write('decoder=CAM_with_AttentionGates\n')
    f.write('ssam=enabled\n')
    f.write(f'boundary_w={BOUNDARY_W}\n')
    f.write(f'batch={BATCH_SIZE}\n')
    f.write(f'lr={LR}\n')
    f.write(f'weight_decay={WEIGHT_DECAY}\n')
    f.write(f'lr_schedule=fixed\n')
    f.write(f'epochs={MAX_EPOCHS}\n')
    f.write(f'img_size={IMG_SIZE}\n')
    f.write(f'seed={SEED}\n')
    f.write(f'classes={NUM_CLASSES}\n')
    f.write(f'loss={LC1}*Dice+{LC2}*CE+{LC3}*BoundaryLoss(w={BOUNDARY_W})_powerset_mutation\n')
print('config.txt saved')

In [ ]:
# Cell 13 — Training loop
# Loss: 0.7*Dice + 0.3*CE + 0.2*BoundaryLoss per 15 non-empty powerset subsets of 4 outputs.
# LR schedule: fixed (paper Section 3.7).
# Checkpoints: last.pth (resumable, every epoch) + best.pth (best val Dice).
from tqdm import tqdm

l  = [0, 1, 2, 3]
ss = [x for x in powerset(l)]
print(f'Powerset subsets: {len(ss)}')

Loss, ValDice = [], []
best_dcs      = 0.80
best_state    = {k: v.cpu().clone() for k, v in net.state_dict().items()}
iter_num      = 0


def do_val():
    net.eval()
    dc_sum = 0.0
    with torch.no_grad():
        for vb in valloader:
            img = vb['image'].squeeze(0).cpu().numpy()
            lbl = vb['label'].squeeze(0).cpu().numpy()
            h, w = img.shape
            if h != IMG_SIZE or w != IMG_SIZE:
                img = zoom(img, (IMG_SIZE / h, IMG_SIZE / w), order=3)
            t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float().cuda()
            with autocast():
                P = net(t)
            out = sum(P)
            out = torch.softmax(out, dim=1).argmax(dim=1).squeeze(0).cpu().numpy()
            if h != IMG_SIZE or w != IMG_SIZE:
                out = zoom(out, (h / IMG_SIZE, w / IMG_SIZE), order=0)
            dc_sum += _dc(out, lbl)
    net.train()
    return dc_sum / len(valloader)


print(f'Training {MAX_EPOCHS} epochs | {len(train_loader)} iters/epoch')
print(f'Snapshot: {snapshot_path}')

for epoch in tqdm(range(MAX_EPOCHS)):
    net.train()
    epoch_loss = 0.0

    for sampled_batch in train_loader:
        imgs   = sampled_batch['image'].type(torch.FloatTensor).cuda()
        labels = sampled_batch['label'].type(torch.FloatTensor).cuda()

        with autocast():
            P    = net(imgs)
            loss = 0.0
            for s in ss:
                if s == []:
                    continue
                iout      = sum(P[idx] for idx in s)
                loss_dice = dice_loss(iout, labels, softmax=True)
                loss_ce   = ce_loss(iout, labels.long())
                loss_bnd  = boundary_loss(iout, labels.long())
                loss     += LC1 * loss_dice + LC2 * loss_ce + LC3 * loss_bnd

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Fixed LR — matches paper Section 3.7 (no schedule)
        iter_num   += 1
        epoch_loss += loss.item()
        if iter_num % 100 == 0:
            tqdm.write(f'  iter {iter_num:5d}  loss {loss.item():.4f}  lr {LR:.2e}')

    Loss.append(epoch_loss / len(train_dataset))

    torch.save({
        'epoch':                epoch,
        'iter_num':             iter_num,
        'model_state_dict':     net.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_dcs':             best_dcs,
    }, os.path.join(snapshot_path, 'last.pth'))

    avg_dcs = do_val()
    ValDice.append(avg_dcs)
    tqdm.write(f'Epoch {epoch+1:3d}/{MAX_EPOCHS}  '
               f'train_loss={Loss[-1]:.5f}  '
               f'val_dc={avg_dcs:.4f}  '
               f'best={best_dcs:.4f}')

    if avg_dcs > best_dcs:
        best_dcs   = avg_dcs
        best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
        torch.save(net.state_dict(), os.path.join(snapshot_path, 'best.pth'))
        tqdm.write(f'  -> New best: {best_dcs:.4f}')

torch.save(best_state, os.path.join(snapshot_path, 'best.pth'))
print(f'Training complete.  Best val Dice: {best_dcs:.4f}')
print(f'Checkpoint: {snapshot_path}/best.pth')

In [ ]:
# Cell 14 — Training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(Loss, color='steelblue')
axes[0].set_title('MIST_CAM (AG + BndLoss) — Train Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ValDice, color='seagreen')
axes[1].axhline(max(ValDice), color='red', linestyle='--', alpha=0.6,
                label=f'Best: {max(ValDice):.4f} @ epoch {ValDice.index(max(ValDice))+1}')
axes[1].set_title('Validation Dice (binary fg)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(snapshot_path, 'training_curves.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')
print(f'Best val Dice: {max(ValDice):.4f}  at epoch {ValDice.index(max(ValDice))+1}')

In [ ]:
# Cell 15 — Package checkpoint for download
import shutil, os
from IPython.display import FileLink, display

zip_base = f'/workspace/MIST_CAM_AG_BndLoss_{IMG_SIZE}_run{run_id}'
print(f'Zipping {snapshot_path} ...')
shutil.make_archive(zip_base, 'zip', snapshot_path)
zip_path = zip_base + '.zip'
size_mb  = os.path.getsize(zip_path) / 1e6
print(f'Created: {zip_path}  ({size_mb:.0f} MB)')
display(FileLink(zip_path))

In [ ]:
# Cell 15b — Optional: upload to Google Drive with rclone
# Configure rclone first: !rclone config
RUN_UPLOAD    = False
REMOTE        = 'gdrive'
GDRIVE_FOLDER = 'MIST_AG_BndLoss_results'

if RUN_UPLOAD:
    import subprocess
    r = subprocess.run(
        ['rclone', 'copy', zip_path, f'{REMOTE}:{GDRIVE_FOLDER}/', '-v'],
        capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print('rclone FAILED:', r.stderr)
else:
    print('Upload skipped (set RUN_UPLOAD=True to enable).')

## Volume-Level Inference
Run after training. Groups the 448 test slices into ~40 patient+frame volumes, runs full 3D inference, and reports **per-class Dice and HD95** — the metrics used in the MIST paper (Table 1).

Paper target (baseline `MIST_CAM`): **Mean Dice 92.56%** — RV 91.23, Myo 90.31, LV 96.14.

In [ ]:
# Cell 16 — Volume-level inference (per-class Dice + HD95, no medpy)
import os, re, glob, sys, collections
import numpy as np
import torch
from tqdm import tqdm
from scipy.ndimage import zoom, binary_erosion, distance_transform_edt

REPO_DIR = '/workspace/MIST'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)

# Metric helpers
def _dc_vol(p, g):
    p, g  = p.astype(bool), g.astype(bool)
    inter = np.count_nonzero(p & g)
    denom = np.count_nonzero(p) + np.count_nonzero(g)
    return 2.0 * inter / float(denom) if denom else 0.0

def _surf_dist(p, g):
    rb = p ^ binary_erosion(p)
    sb = g ^ binary_erosion(g)
    return distance_transform_edt(~g)[rb], distance_transform_edt(~p)[sb]

def _hd95_vol(p, g):
    p, g = p.astype(bool), g.astype(bool)
    if not p.any() or not g.any():
        return 0.0
    d1, d2 = _surf_dist(p, g)
    return float(np.percentile(np.hstack([d1, d2]), 95))

# Load model
CHECKPOINT = os.path.join(snapshot_path, 'best.pth')
print(f'Loading: {CHECKPOINT}')

from lib.networks import MIST_CAM as _MIST_CAM
net_inf = _MIST_CAM(
    n_class=NUM_CLASSES,
    img_size_s1=(IMG_SIZE, IMG_SIZE),
    img_size_s2=(224, 224),
    model_scale='small',
    decoder_aggregation='additive',
    interpolation='bilinear',
).cuda()
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
net_inf.load_state_dict(state)
net_inf.eval()
print('Model loaded')

# Group test slices by patient+frame volume
test_npz = sorted(glob.glob(os.path.join(TEST_DIR, '*.npz')))
print(f'Test slices found: {len(test_npz)}')

vol_dict = collections.defaultdict(list)
for path in test_npz:
    fname = os.path.basename(path)
    m = re.match(r'(patient\d+_frame\d+)_slice(\d+)\.npz', fname)
    if m:
        vol_dict[m.group(1)].append((int(m.group(2)), path))
    else:
        print(f'  Unrecognised filename: {fname}')
print(f'Volumes found: {len(vol_dict)}')

# Per-volume inference and metrics
all_metrics = []

with torch.no_grad():
    for vol_key, slice_list in tqdm(sorted(vol_dict.items()), desc='Volumes'):
        slice_list.sort(key=lambda x: x[0])

        imgs, lbls = [], []
        for _, path in slice_list:
            d = np.load(path)
            imgs.append(d['image'])
            lbls.append(d['label'])

        img_vol  = np.stack(imgs)
        lbl_vol  = np.stack(lbls)
        pred_vol = np.zeros_like(lbl_vol)

        for si in range(len(slice_list)):
            slc = img_vol[si]
            h, w = slc.shape
            if h != IMG_SIZE or w != IMG_SIZE:
                slc = zoom(slc, (IMG_SIZE / h, IMG_SIZE / w), order=3)
            t = torch.from_numpy(slc).unsqueeze(0).unsqueeze(0).float().cuda()
            with torch.cuda.amp.autocast():
                P = net_inf(t)
            out = sum(P)
            out = torch.softmax(out, dim=1).argmax(dim=1).squeeze(0).cpu().numpy()
            if h != IMG_SIZE or w != IMG_SIZE:
                out = zoom(out, (h / IMG_SIZE, w / IMG_SIZE), order=0)
            pred_vol[si] = out

        vol_metrics = []
        for c in range(1, NUM_CLASSES):  # skip background
            p_c = (pred_vol == c)
            g_c = (lbl_vol  == c)
            vol_metrics.append((_dc_vol(p_c, g_c), _hd95_vol(p_c, g_c)))
        all_metrics.append(vol_metrics)

M = np.array(all_metrics)  # [N_vols, 3, 2]
class_names = ['RV', 'Myo', 'LV']
print(f'\n=== Volume-Level Results  ({len(all_metrics)} volumes) ===')
for ci, cname in enumerate(class_names):
    dc_mean = M[:, ci, 0].mean() * 100
    hd_mean = M[:, ci, 1].mean()
    print(f'  Class {ci+1} ({cname:<3})  Dice = {dc_mean:.2f}%   HD95 = {hd_mean:.2f} mm')
mean_dice = M[:, :, 0].mean() * 100
print(f'  Mean Dice  : {mean_dice:.2f}%')
print(f'  (paper baseline: 92.56%)')